<a href="https://colab.research.google.com/github/ilkaydemirhan/Quran-semantic-search/blob/main/kuran_anlamsal_arama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [12]:
import requests
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("1. Adım: Güncel Kur'an verileri (Arapça ve Türkçe Meal) API üzerinden indiriliyor...")

# Hem Arapça orijinal metni hem de Türkçe Diyanet mealini aynı anda çekmek için iki kaynağı birleştiriyoruz
url_arapca = "http://api.alquran.cloud/v1/quran/ar.jalalayn" # veya quran-uthmani
url_turkce = "http://api.alquran.cloud/v1/quran/tr.diyanet"

resp_tr = requests.get(url_turkce).json()
resp_ar = requests.get(url_arapca).json()

tum_ayetler = []

# Türkçe ve Arapça listeleri sure ve ayet numaralarına göre eşleştiriyoruz
for i, sure_tr in enumerate(resp_tr['data']['surahs']):
    sure_ar = resp_ar['data']['surahs'][i]
    sure_adi = sure_tr['englishName']
    sure_islemeli = sure_tr['name']

    for j, ayet_tr in enumerate(sure_tr['ayahs']):
        ayet_ar = sure_ar['ayahs'][j]

        tum_ayetler.append({
            "sure": sure_islemeli,
            "sure_en": sure_adi,
            "ayet_no": ayet_tr['numberInSurah'],
            "arapca": ayet_ar['text'],
            "meal": ayet_tr['text']
        })

df_kuran = pd.DataFrame(tum_ayetler)
print(f"Toplam {len(df_kuran)} ayet başarıyla hafızaya alındı.")

print("\n2. Adım: Yapay zeka modeli yükleniyor ve vektörleştirme başlatılıyor (Bu işlem birkaç saniye sürebilir)...")
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Aramanın hem Arapça hem Türkçe anlam üzerinden yapılabilmesi için metinleri birleştiriyoruz
metinler = (df_kuran["arapca"] + " - " + df_kuran["meal"]).tolist()
vektorler = model.encode(metinler, show_progress_bar=True)

print("\nTüm Kur'an başarıyla vektörleştirildi! Artık arama yapmaya hazırsınız.")

1. Adım: Güncel Kur'an verileri (Arapça ve Türkçe Meal) API üzerinden indiriliyor...
Toplam 6236 ayet başarıyla hafızaya alındı.

2. Adım: Yapay zeka modeli yükleniyor ve vektörleştirme başlatılıyor (Bu işlem birkaç saniye sürebilir)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/195 [00:00<?, ?it/s]


Tüm Kur'an başarıyla vektörleştirildi! Artık arama yapmaya hazırsınız.


In [19]:
# Aramak istediğiniz kavramı buraya yazın (Örn: "merhamet", "cennet ve cehennem", "sabır", "adalet")
arama_sorgusu = "sabır"

# Sorguyu vektöre çeviriyoruz
sorgu_vektoru = model.encode([arama_sorgusu])

# Tüm Kur'an vektörleri ile benzerlikleri hesaplıyoruz
benzerlikler = cosine_similarity(sorgu_vektoru, vektorler)[0]

# En yüksek benzerlik skoruna sahip ilk 3 ayeti buluyoruz
en_iyi_3_index = np.argsort(benzerlikler)[::-1][:10]

print(f"=== Aradığınız Kavram: '{arama_sorgusu}' ===\n")

for sira, idx in enumerate(en_iyi_3_index, 1):
    ayet = df_kuran.iloc[idx]
    skor = benzerlikler[idx]
    print(f"{sira}. Sonuç (Benzerlik: {skor:.2f})")
    print(f"Sure: {ayet['sure']} ({ayet['sure_en']}) - Ayet {ayet['ayet_no']}")
    print(f"Arapça: {ayet['arapca']}")
    print(f"Meal: {ayet['meal']}")
    print("-" * 50)

=== Aradığınız Kavram: 'sabır' ===

1. Sonuç (Benzerlik: 0.48)
Sure: سُورَةُ هُودٍ (Hud) - Ayet 122
Arapça: «وانتظروا» عاقبة أمركم «إنا منتظرون» ذلك.
Meal: İnanmayanlara: "Durumunuzun gerektirdiğini yapın, doğrusu biz de yapıyoruz; bekleyin, biz de bekliyoruz" de.
--------------------------------------------------
2. Sonuç (Benzerlik: 0.47)
Sure: سُورَةُ الشُّورَىٰ (Ash-Shura) - Ayet 43
Arapça: «ولمن صبر» فلم ينتصر «وغفر» تجاوز «إن ذلك» الصبر والتجاوز «لمن عزم الأمور» أي معزوماتها، بمعنى المطلوبات شرعاً.
Meal: Ama sabredip bağışlayanın işi, işte bu, azmedilmeye değer işlerdendir.
--------------------------------------------------
3. Sonuç (Benzerlik: 0.43)
Sure: سُورَةُ المُدَّثِّرِ (Al-Muddaththir) - Ayet 55
Arapça: «فمن شاء ذكره» قرأه فاتعظ به.
Meal: Dileyen kimse öğüt alır.
--------------------------------------------------
4. Sonuç (Benzerlik: 0.43)
Sure: سُورَةُ الفَجۡرِ (Al-Fajr) - Ayet 30
Arapça: «وادخلي جنتي» معهم.
Meal: Cennetime gir.
------------------------------------------